# Lab 7 — Tools (Features #16–#22)
**AmbariCloud FDE Workshop · Day 1** · Work inside `~/workshop-todo`.

## Feature #16 — Built-in tools
| Tool | Does |
|---|---|
| Read / Write / Edit | open · create · surgical find-and-replace |
| Bash | run shell — tests, git, installs |
| Grep / Glob | search content · find files by pattern |
| WebFetch, WebSearch, Agent, NotebookEdit | also built in |

**Reflex:** Grep/Glob to FIND → Read only what you'll change → Edit → Bash to run.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> where is the JSON file written?                    (watch: Grep)
> rename todos.json to data/todos.json everywhere    (watch: Glob, Read, Edit, Bash mkdir)
> run the tests                                      (watch: Bash pytest)
```
**Should see:** tool names in the transcript; permission prompt for Write/Edit/Bash; Read/Grep/Glob run without asking.

## Feature #17 — `!` shell mode
Prefix a line with `!` to run a command yourself; output feeds into the conversation.
```
> ! pytest -q
> ! git status
> ! python todo.py add "demo" && python todo.py list
> the last list output looks wrong — fix the formatting
```
**Watch for:** long outputs land in context in full — pipe through tail/grep. `!` runs with YOUR permissions, no Claude gate.


## Feature #18 — Hooks: deterministic guardrails
Shell commands that fire on tool events: `PreToolUse`, `PostToolUse`, `UserPromptSubmit`, `Stop`, `SessionStart`, `Notification`…
**DETERMINISTIC** — unlike a CLAUDE.md line, a hook ALWAYS runs. `PreToolUse` can block (exit 2 + message).

`.claude/settings.json`:
```json
{ "hooks": { "PostToolUse": [ {
   "matcher": "Edit|Write",
   "hooks": [ { "type": "command",
     "command": "ruff format $(jq -r .tool_input.file_path) >/dev/null" } ] } ] } }
```
Then `/hooks` to confirm registration; `> add a docstring to todo.py` → file gets auto-formatted.
**Watch for:** hook stdin is JSON (tool name, input, session). Test the command standalone first — a broken hook silently fails every edit.


### Write the hook file (runnable)

In [ ]:
%%bash
mkdir -p ~/workshop-todo/.claude
cat > ~/workshop-todo/.claude/settings.json <<'EOF'
{ "hooks": { "PostToolUse": [ {
   "matcher": "Edit|Write",
   "hooks": [ { "type": "command",
     "command": "ruff format $(jq -r .tool_input.file_path) >/dev/null 2>&1 || true" } ] } ] } }
EOF
cat ~/workshop-todo/.claude/settings.json

## Feature #19 — MCP servers: give your agent hands
Model Context Protocol: GitHub, Playwright, Slack, databases, your APIs.
Transports: stdio (local), HTTP, SSE. Scopes: local · project (`.mcp.json`, shared) · user.
```bash
claude mcp add playwright -- npx @playwright/mcp@latest
claude mcp add --transport http github https://api.githubcopilot.com/mcp/
claude mcp list
```
In session: `/mcp` (status, auth, enable/disable), then
`> open https://example.com and tell me the <h1>` → tool call named `mcp__playwright__browser_navigate`.
**Watch for:** MCP tool schemas count against context — add the servers you use, not all of them.

## Feature #20 — Subagents: isolated, focused, parallel
Agents with their OWN context window; only the conclusion returns. Custom: `.claude/agents/<name>.md` with YAML frontmatter (name, description, tools, model). Manage with `/agents`.
> ⌨️ **Interactive step — do this in a real terminal**, not in this notebook. Claude Code is a TUI; the notebook cells below only run *headless* (`claude -p`) commands.

```
> /agents -> Create new agent -> Project
#  name: test-writer
#  description: writes pytest tests for a file
#  tools: Read, Write, Bash
> use the test-writer agent to cover todo.py
> use an Explore agent to find every place todos.json is referenced
```
**Watch for:** a subagent cannot see your chat — put everything it needs in the prompt. Least privilege (reviewer = Read only).


## Feature #21 — Skills: reusable procedures (`/name`)
`SKILL.md` procedures Claude auto-invokes when relevant — or run with `/name`.
Live in `.claude/skills/<name>/SKILL.md` (project) or `~/.claude/skills/` (user). Load ONLY when used — zero context cost until invoked.


### Create a /ship skill (runnable)

In [ ]:
%%bash
mkdir -p ~/workshop-todo/.claude/skills/ship
cat > ~/workshop-todo/.claude/skills/ship/SKILL.md <<'EOF'
---
name: ship
description: run tests, then commit with a clear message
---
1. Run pytest -q; stop if red.
2. git add -A; commit; summarise what shipped.
EOF
cat ~/workshop-todo/.claude/skills/ship/SKILL.md

Then in the TUI: `/ship` appears in the `/` menu instantly (no restart). Saying **"ship it"** without the slash also triggers it — auto-invocation.
**Watch for:** frontmatter description is the trigger — write it as WHEN to use. Arguments arrive as `$ARGUMENTS`.

## Feature #22 — Plugins: package and share everything
One installable bundle of skills + hooks + MCP servers + subagents. Distributed through marketplaces (a git repo with `marketplace.json`).
```bash
claude plugin marketplace add anthropics/claude-plugins-official
claude plugin install code-review@claude-plugins-official
claude --plugin-dir ./my-plugin      # test your own locally
```
In session: `/plugin` (Discover / Installed / Marketplaces).
**Watch for:** plugins run hooks and MCP servers on your machine — trust the source like a package registry.
